# Huấn luyện và Fine-tune PhoBERT cho phân tích cảm xúc bình luận YouTube
**Dự án:** YTB_sentiment_analysis  
**Thành viên phụ trách:** Bạn Hùng (Deep Learning & PhoBERT)  
**Mục tiêu:** Cấu hình PhoBERT Tokenizer, Fine-tune PhoBERT trên GPU Google Colab, đánh giá chỉ số ($Accuracy, F1$) so sánh với SVM, xuất file trọng số lưu lên Google Drive chung.

## 1. Kiểm tra cấu hình GPU và Cài đặt môi trường

In [ ]:
!nvidia-smi
!pip install -q transformers torch accelerate scikit-learn pandas numpy matplotlib seaborn underthesea pyvi tqdm

## 2. Kết nối Google Drive và chuẩn bị dữ liệu sạch từ Vitou
Tải dữ liệu `train.csv` và `test.csv` từ Google Drive chung của nhóm vào Colab.

In [ ]:
# Chuẩn bị thư mục dữ liệu trên Colab
import os
import pandas as pd
from google.colab import files

LOCAL_DATA_DIR = './data'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# Kiểm tra nếu chưa có file thì mở hộp thoại upload trực tiếp từ máy tính
if not os.path.exists(f'{LOCAL_DATA_DIR}/train.csv'):
    print("Chưa thấy dữ liệu, vui lòng bấm nút bên dưới và chọn 2 file 'train.csv' và 'test.csv' từ máy bạn:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        os.rename(filename, f"{LOCAL_DATA_DIR}/{filename}")

print("Dữ liệu đã sẵn sàng:")
df_train = pd.read_csv(f'{LOCAL_DATA_DIR}/train.csv')
df_test = pd.read_csv(f'{LOCAL_DATA_DIR}/test.csv')
print(f"- Train samples: {len(df_train):,}")
print(f"- Test samples : {len(df_test):,}")

## 3. Clone Repository Git của nhóm & Chuyển sang nhánh của Hùng

In [ ]:
# Clone repo nếu chạy độc lập trên Colab
!git clone https://github.com/dung06coder/YTB_sentiment_analysis.git
%cd YTB_sentiment_analysis
!git checkout feature/hung-phobert-dl

# Đảm bảo thư mục data chứa dữ liệu vừa tải
!mkdir -p machine_learning/data
!cp /content/data/*.csv machine_learning/data/

## 4. Tiến hành Huấn luyện (Fine-tune) PhoBERT trên GPU (CUDA + FP16)
Chạy script `train_phobert.py` đã được cấu hình tối ưu với:
- PhoBERT Base (`vinai/phobert-base`)
- Tokenizer hỗ trợ từ tiếng Việt có phân đoạn từ (`segment=True`)
- Batch size 32, Learning Rate 2e-5, AdamW + Linear Warmup
- Tự động chọn checkpoint tốt nhất theo Macro F1

In [ ]:
!python machine_learning/train_phobert.py \
    --train machine_learning/data/train.csv \
    --test machine_learning/data/test.csv \
    --epochs 4 \
    --batch-size 32 \
    --lr 2e-5 \
    --max-length 128 \
    --fp16

## 5. Đánh giá và Vẽ biểu đồ so sánh hiệu năng với SVM Baseline
Chạy `evaluate_and_compare.py` để xuất:
1. Biểu đồ so sánh Accuracy & F1-score (`comparison_metrics_barchart.png`)
2. Biểu đồ F1 từng lớp (`f1_per_class_comparison.png`)
3. Ma trận nhầm lẫn PhoBERT (`phobert_confusion_matrix.png`)
4. Bảng mã nguồn LaTeX chép vào báo cáo đồ án (`latex_comparison_table.tex`)
5. Báo cáo phân tích chuyên sâu (`comparison_report.md`)

In [ ]:
!python machine_learning/evaluate_and_compare.py

# Hiển thị biểu đồ ngay trên Colab
from IPython.display import Image, display
display(Image('machine_learning/reports/comparison_metrics_barchart.png'))
display(Image('machine_learning/reports/f1_per_class_comparison.png'))
display(Image('machine_learning/reports/phobert_confusion_matrix.png'))

## 6. Nén trọng số mô hình và lưu vào Google Drive chung
> **LƯU Ý NGHIÊM NGẶT:** File trọng số PhoBERT (`.safetensors` hoặc `.bin` dung lượng ~540MB) **TUYỆT ĐỐI CẤM** đẩy lên GitHub. Hãy nén và đẩy vào Google Drive chung của nhóm để bạn làm Backend tải về tích hợp.

In [ ]:
# Nén thư mục model đã fine-tune
!zip -r phobert_finetuned.zip machine_learning/models/phobert_finetuned

# Tải thẳng vào Google Drive chung của nhóm
DRIVE_MODELS_DIR = '/content/drive/MyDrive/YTB_sentiment_analysis/models'
!mkdir -p {DRIVE_MODELS_DIR}
!cp phobert_finetuned.zip {DRIVE_MODELS_DIR}/
print(f"Đã lưu thành công mô hình lên Google Drive: {DRIVE_MODELS_DIR}/phobert_finetuned.zip")

## 7. Kiểm thử nhanh hàm suy luận (Inference Demo)

In [ ]:
from machine_learning.predict_phobert import PhoBertSentimentPredictor

predictor = PhoBertSentimentPredictor('machine_learning/models/phobert_finetuned')

sample_comments = [
    "Video phân tích quá chi tiết và hữu ích, cảm ơn anh nhiều! <3",
    "Nội dung nhảm nhí, giật tít câu view phí thời gian 😡",
    "Bài hát ở phút 3:45 tên là gì vậy mọi người?",
    "Lúc đầu xem tưởng dở ai dè hay vãi chưởng kkkk"
]

for cmt in sample_comments:
    res = predictor.predict_comment(cmt)
    print(f"Bình luận: {res['text']}")
    print(f"--> Cảm xúc: {res['label'].upper()} (Độ tin cậy: {res['confidence']:.2%})")
    print(f"    Xác suất: {res['probabilities']}\n")